In [ ]:
!pip install -q datasets soundfile transformers accelerate torch resemblyzer librosa pandas scipy

In [ ]:
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import os
from pathlib import Path
from scipy.signal import resample
from transformers import VitsModel, AutoTokenizer
from resemblyzer import VoiceEncoder, preprocess_wav
import glob

# Updated Config
NUM_SAMPLES = 30
MODEL_NAME = "facebook/mms-tts-urd-script_arabic"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading models to {device}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = VitsModel.from_pretrained(MODEL_NAME).to(device)
model.eval()
encoder = VoiceEncoder()

def run_evaluation(dataset_name, samples_list, text_key):
    base_dir = f"mushra_{dataset_name}"
    for sub in ["reference", "generated", "anchor"]:
        os.makedirs(f"{base_dir}/{sub}", exist_ok=True)

    results = []
    for i in range(min(NUM_SAMPLES, len(samples_list))):
        try:
            sample = samples_list[i]
            text = sample[text_key]
            ref_array = np.array(sample['audio']['array']).astype(np.float32)
            ref_sr = sample['audio']['sampling_rate']

            if ref_array.ndim > 1:
                ref_array = ref_array.mean(axis=1)

            ref_path = f"{base_dir}/reference/sample_{i}.wav"
            gen_path = f"{base_dir}/generated/sample_{i}.wav"
            anc_path = f"{base_dir}/anchor/sample_{i}.wav"

            # 1. Save Reference
            sf.write(ref_path, ref_array, ref_sr)

            # 2. Generate TTS
            inputs = tokenizer(text, return_tensors="pt").to(device)
            with torch.no_grad():
                output = model(**inputs)
            gen_speech = output.waveform[0].cpu().numpy()
            gen_sr = model.config.sampling_rate
            sf.write(gen_path, gen_speech, gen_sr)

            # 3. Create Anchor (8kHz resample cycle)
            target_len_8k = int(len(ref_array) * 8000 / ref_sr)
            downsampled = resample(ref_array, target_len_8k)
            upsampled = resample(downsampled, len(ref_array))
            sf.write(anc_path, upsampled, ref_sr)

            # 4. Metrics
            wav_ref = preprocess_wav(Path(ref_path))
            wav_gen = preprocess_wav(Path(gen_path))

            emb_ref_full = encoder.embed_utterance(wav_ref)
            emb_gen = encoder.embed_utterance(wav_gen)

            sim = np.dot(emb_ref_full, emb_gen) / (np.linalg.norm(emb_ref_full) * np.linalg.norm(emb_gen))

            results.append({
                "sample_id": i,
                "urdu_text": text,
                "ref_gen_similarity": float(sim)
            })
            print(f"[{dataset_name.upper()}] Sample {i+1}/{NUM_SAMPLES}: Similarity={sim:.4f}")

        except Exception as e:
            print(f"Error in {dataset_name} sample {i}: {e}")
            continue

    return results

Loading models to cuda...


Loading weights:   0%|          | 0/762 [00:00<?, ?it/s]

Loaded the voice encoder model on cuda in 0.09 seconds.


In [ ]:
import zipfile
import glob
import os
from google.colab import drive
import pandas as pd
import numpy as np
import soundfile as sf

# 1. Mount Google Drive
print("🔄 Mounting Google Drive...")
drive.mount('/content/drive')

# 2. Path to ZIP
zip_path = "/content/drive/MyDrive/urduser.zip"

if not os.path.exists(zip_path):
    print("❌ ZIP not found at that path!")
else:
    print(f"✅ Found ZIP: {zip_path}")

    # 3. Unzip
    extract_path = "urduser_data"
    os.makedirs(extract_path, exist_ok=True)
    print("📦 Unzipping...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"✅ Unzipped to {extract_path}/")

    # 4. Find metadata Excel
    excel_files = glob.glob(f"{extract_path}/**/*.xlsx", recursive=True)
    if not excel_files:
        raise FileNotFoundError("❌ Could not find .xlsx metadata file!")
    metadata_path = excel_files[0]

    # 5. Load metadata - Skipping the first header row based on observed data structure
    df = pd.read_excel(metadata_path, skiprows=1)
    print("✅ Metadata loaded. Columns:", df.columns.tolist())

    # Mapping based on the Urdu headers observed in the error state
    FILENAME_COL = "فائل کا نام"
    TEXT_COL = "تشریح"

    # 6. Create samples
    urduser_samples = []
    for idx, row in df.head(NUM_SAMPLES).iterrows():
        file_name = str(row[FILENAME_COL])
        if not file_name.endswith('.wav'):
            file_name += '.wav'

        # Find audio file
        audio_paths = glob.glob(f"{extract_path}/**/{file_name}", recursive=True)
        if not audio_paths:
            continue

        audio_path = audio_paths[0]
        ref_array, ref_sr = sf.read(audio_path)
        if ref_array.ndim > 1:
            ref_array = np.mean(ref_array, axis=1)

        urduser_samples.append({
            "audio": {
                "array": ref_array.astype(np.float32),
                "sampling_rate": int(ref_sr)
            },
            "transcription": str(row[TEXT_COL])
        })

    print(f"✅ Loaded {len(urduser_samples)} samples from UrduSER")

    # 7. Run evaluation
    results_urduser = run_evaluation("urduser", urduser_samples, text_key="transcription")

    # 8. Save results
    df_urduser = pd.DataFrame(results_urduser)
    df_urduser.to_csv("urduser_evaluation_results.csv", index=False)
    print("\n🎉 UrduSER Evaluation Complete!")

🔄 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Found ZIP: /content/drive/MyDrive/urduser.zip
📦 Unzipping...
✅ Unzipped to urduser_data/
✅ Metadata loaded. Columns: ['تشریح', 'جذبات', 'صنف', 'اداکار', 'اداکار  کاآئی ڈی نمبر', 'نمونہ کی شکل', 'نمونہ کی شرح', 'فارمیٹ', 'دورانیہ سیکنڈز میں ', 'فائل کا نام', 'سیریل نمبر']
✅ Loaded 30 samples from UrduSER
[URDUSER] Sample 1/30: Similarity=0.6270
[URDUSER] Sample 2/30: Similarity=0.4968
[URDUSER] Sample 3/30: Similarity=0.5136
[URDUSER] Sample 4/30: Similarity=0.5180
[URDUSER] Sample 5/30: Similarity=0.5462
[URDUSER] Sample 6/30: Similarity=0.6020
[URDUSER] Sample 7/30: Similarity=0.5884
[URDUSER] Sample 8/30: Similarity=0.5831
[URDUSER] Sample 9/30: Similarity=0.5054
[URDUSER] Sample 10/30: Similarity=0.5686
[URDUSER] Sample 11/30: Similarity=0.5778
[URDUSER] Sample 12/30: Similarity=0.3414
[URDUSER] Sample 13/30: Similarity=0.5747


In [ ]:
import shutil
from google.colab import files

# Zip the results folder
shutil.make_archive("mushra_urduser_30samples", 'zip', "mushra_urduser")

# Download
files.download("mushra_urduser_30samples.zip")
files.download("urduser_evaluation_results.csv")
print("✅ All files downloaded!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ All files downloaded!


**Reasoning**:
Load the UrduSER metadata from the previously unzipped directory to inspect unique emotion categories and their sample counts.



In [1]:
import pandas as pd
import glob
import os

# 1. Locate the metadata file
metadata_files = glob.glob('urduser_data/**/*.xlsx', recursive=True)
if not metadata_files:
    print('Error: Could not find the metadata .xlsx file.')
else:
    metadata_path = metadata_files[0]
    print(f'Loading metadata from: {metadata_path}')

    # 2. Load the metadata skipping the first row (based on previous exploration)
    df_metadata = pd.read_excel(metadata_path, skiprows=1)

    # 3. Define the emotion column name
    emotion_col = 'جذبات'

    # 4. Identify unique categories and their counts
    unique_emotions = df_metadata[emotion_col].unique()
    emotion_counts = df_metadata[emotion_col].value_counts()

    print('\nUnique Emotion Categories:')
    print(unique_emotions)
    print('\nCounts per Emotion Category:')
    print(emotion_counts)

    # Display first few rows to confirm structure
    display(df_metadata.head())

Error: Could not find the metadata .xlsx file.


**Reasoning**:
The previous attempt to find the metadata file failed because it looked in 'urduser_data', but the previous extraction cell (ur4MnQew6vHR) showed that files might be nested or the directory structure might be different. I will check the current working directory to locate the correct path to the unzipped metadata.



In [3]:
import os
import glob
import pandas as pd

# 1. Broad search for any xlsx file to find the correct path
all_xlsx = glob.glob('**/*.xlsx', recursive=True)
print(f"All Excel files found: {all_xlsx}")

if not all_xlsx:
    print("No Excel files found. Checking directory structure...")
    !ls -R urduser_data | head -n 20
else:
    metadata_path = all_xlsx[0]
    print(f"Using metadata file: {metadata_path}")

    # 2. Load the metadata
    # Using the structure confirmed in cell ur4MnQew6vHR
    df_metadata = pd.read_excel(metadata_path, skiprows=1)

    # 3. Analyze emotions
    emotion_col = 'جذبات'
    if emotion_col in df_metadata.columns:
        unique_emotions = df_metadata[emotion_col].unique()
        emotion_counts = df_metadata[emotion_col].value_counts()

        print('\nUnique Emotion Categories:')
        print(unique_emotions)
        print('\nCounts per Emotion Category:')
        print(emotion_counts)

        # Store for next subtask
        %store df_metadata
    else:
        print(f"Column '{emotion_col}' not found. Available columns: {df_metadata.columns.tolist()}")

    display(df_metadata.head())

All Excel files found: []
No Excel files found. Checking directory structure...
ls: cannot access 'urduser_data': No such file or directory
